In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName("SCNE Feature Engineering") \
    .master("local[*]") \
    .getOrCreate()

DATA_PATH = r"D:\Big Data Programming Project\Final Assignment\data\processed\spark\scne_multiday_delay"

df = spark.read.parquet(DATA_PATH)

print("Rows:", df.count())
print("Columns:", len(df.columns))

Rows: 308885
Columns: 35


In [2]:
feature_df = (
    df
    .withColumn("hour", F.hour("scheduled_arrival_time"))
    .withColumn("minute", F.minute("scheduled_arrival_time"))
    .withColumn("day_of_week", F.dayofweek("service_date"))
    .withColumn(
        "is_weekend",
        F.when(F.dayofweek("service_date").isin([1, 7]), 1).otherwise(0)
    )
    .withColumn(
        "is_public_holiday",
        F.when(F.col("service_date") == F.lit("2025-12-26"), 1).otherwise(0)
    )
)

feature_df.select(
    "service_date",
    "hour",
    "minute",
    "day_of_week",
    "is_weekend",
    "is_public_holiday"
).show(10)

+------------+----+------+-----------+----------+-----------------+
|service_date|hour|minute|day_of_week|is_weekend|is_public_holiday|
+------------+----+------+-----------+----------+-----------------+
|  2025-12-26|  17|    14|          6|         0|                1|
|  2025-12-26|  16|    28|          6|         0|                1|
|  2025-12-26|  16|    23|          6|         0|                1|
|  2025-12-26|  19|    18|          6|         0|                1|
|  2025-12-26|  16|     1|          6|         0|                1|
|  2025-12-26|  17|     0|          6|         0|                1|
|  2025-12-26|  19|    54|          6|         0|                1|
|  2025-12-26|  21|    26|          6|         0|                1|
|  2025-12-26|  18|     4|          6|         0|                1|
|  2025-12-26|  23|    47|          6|         0|                1|
+------------+----+------+-----------+----------+-----------------+
only showing top 10 rows



In [3]:
trip_window = Window.partitionBy("service_date", "trip_id")

feature_df = (
    feature_df
    .withColumn(
        "max_stop_sequence",
        F.max("stop_sequence").over(trip_window)
    )
    .withColumn(
        "journey_progress",
        F.when(
            F.col("max_stop_sequence") > 0,
            F.col("stop_sequence") / F.col("max_stop_sequence")
        ).otherwise(0)
    )
)

feature_df.select(
    "trip_id",
    "stop_sequence",
    "max_stop_sequence",
    "journey_progress"
).show(10, truncate=False)

+------------------------------------------+-------------+-----------------+-------------------+
|trip_id                                   |stop_sequence|max_stop_sequence|journey_progress   |
+------------------------------------------+-------------+-----------------+-------------------+
|VJ0078a6f9e41d27c42ba12ee0c3d1f99b0052e788|15           |37               |0.40540540540540543|
|VJ0078a6f9e41d27c42ba12ee0c3d1f99b0052e788|28           |37               |0.7567567567567568 |
|VJ0078a6f9e41d27c42ba12ee0c3d1f99b0052e788|2            |37               |0.05405405405405406|
|VJ0078a6f9e41d27c42ba12ee0c3d1f99b0052e788|16           |37               |0.43243243243243246|
|VJ0078a6f9e41d27c42ba12ee0c3d1f99b0052e788|14           |37               |0.3783783783783784 |
|VJ0078a6f9e41d27c42ba12ee0c3d1f99b0052e788|32           |37               |0.8648648648648649 |
|VJ0078a6f9e41d27c42ba12ee0c3d1f99b0052e788|23           |37               |0.6216216216216216 |
|VJ0078a6f9e41d27c42ba12ee0c3d

In [4]:
journey_order = Window.partitionBy(
    "service_date",
    "trip_id"
).orderBy("stop_sequence")

feature_df = feature_df.withColumn(
    "previous_stop_delay",
    F.lag("delay_seconds", 1).over(journey_order)
)

feature_df.select(
    "trip_id",
    "stop_sequence",
    "delay_seconds",
    "previous_stop_delay"
).show(15, truncate=False)

+------------------------------------------+-------------+-------------+-------------------+
|trip_id                                   |stop_sequence|delay_seconds|previous_stop_delay|
+------------------------------------------+-------------+-------------+-------------------+
|VJ0078a6f9e41d27c42ba12ee0c3d1f99b0052e788|0            |139.0        |NULL               |
|VJ0078a6f9e41d27c42ba12ee0c3d1f99b0052e788|1            |159.0        |139.0              |
|VJ0078a6f9e41d27c42ba12ee0c3d1f99b0052e788|2            |179.0        |159.0              |
|VJ0078a6f9e41d27c42ba12ee0c3d1f99b0052e788|3            |280.0        |179.0              |
|VJ0078a6f9e41d27c42ba12ee0c3d1f99b0052e788|4            |320.0        |280.0              |
|VJ0078a6f9e41d27c42ba12ee0c3d1f99b0052e788|5            |360.0        |320.0              |
|VJ0078a6f9e41d27c42ba12ee0c3d1f99b0052e788|7            |220.0        |360.0              |
|VJ0078a6f9e41d27c42ba12ee0c3d1f99b0052e788|8            |281.0       

In [5]:
history_window = (
    Window.partitionBy("service_date", "trip_id")
    .orderBy("stop_sequence")
    .rowsBetween(-3, -1)
)

feature_df = feature_df.withColumn(
    "rolling_previous_delay",
    F.avg("delay_seconds").over(history_window)
)

feature_df.select(
    "trip_id",
    "stop_sequence",
    "delay_seconds",
    "previous_stop_delay",
    "rolling_previous_delay"
).show(15, truncate=False)

+------------------------------------------+-------------+-------------+-------------------+----------------------+
|trip_id                                   |stop_sequence|delay_seconds|previous_stop_delay|rolling_previous_delay|
+------------------------------------------+-------------+-------------+-------------------+----------------------+
|VJ0078a6f9e41d27c42ba12ee0c3d1f99b0052e788|0            |139.0        |NULL               |NULL                  |
|VJ0078a6f9e41d27c42ba12ee0c3d1f99b0052e788|1            |159.0        |139.0              |139.0                 |
|VJ0078a6f9e41d27c42ba12ee0c3d1f99b0052e788|2            |179.0        |159.0              |149.0                 |
|VJ0078a6f9e41d27c42ba12ee0c3d1f99b0052e788|3            |280.0        |179.0              |159.0                 |
|VJ0078a6f9e41d27c42ba12ee0c3d1f99b0052e788|4            |320.0        |280.0              |206.0                 |
|VJ0078a6f9e41d27c42ba12ee0c3d1f99b0052e788|5            |360.0        |

In [6]:
feature_df = (
    feature_df
    .withColumn(
        "has_previous_delay",
        F.when(F.col("previous_stop_delay").isNotNull(), 1).otherwise(0)
    )
    .fillna({
        "previous_stop_delay": 0.0,
        "rolling_previous_delay": 0.0
    })
)

feature_df.select(
    "stop_sequence",
    "delay_seconds",
    "previous_stop_delay",
    "rolling_previous_delay",
    "has_previous_delay"
).show(12)

+-------------+-------------+-------------------+----------------------+------------------+
|stop_sequence|delay_seconds|previous_stop_delay|rolling_previous_delay|has_previous_delay|
+-------------+-------------+-------------------+----------------------+------------------+
|            0|        139.0|                0.0|                   0.0|                 0|
|            1|        159.0|              139.0|                 139.0|                 1|
|            2|        179.0|              159.0|                 149.0|                 1|
|            3|        280.0|              179.0|                 159.0|                 1|
|            4|        320.0|              280.0|                 206.0|                 1|
|            5|        360.0|              320.0|     259.6666666666667|                 1|
|            7|        220.0|              360.0|                 320.0|                 1|
|            8|        281.0|              220.0|                 300.0|        

In [7]:
model_df = feature_df.select(
    "service_date",
    "trip_id",
    "published_line_name",
    "direction_id",
    "stop_id",
    "stop_sequence",
    "hour",
    "minute",
    "day_of_week",
    "is_weekend",
    "is_public_holiday",
    "journey_progress",
    "previous_stop_delay",
    "rolling_previous_delay",
    "has_previous_delay",
    "delay_seconds"
)

print("Rows:", model_df.count())
print("Columns:", len(model_df.columns))

model_df.show(5, truncate=False)

Rows: 308885
Columns: 16
+------------+------------------------------------------+-------------------+------------+------------+-------------+----+------+-----------+----------+-----------------+-------------------+-------------------+----------------------+------------------+-------------+
|service_date|trip_id                                   |published_line_name|direction_id|stop_id     |stop_sequence|hour|minute|day_of_week|is_weekend|is_public_holiday|journey_progress   |previous_stop_delay|rolling_previous_delay|has_previous_delay|delay_seconds|
+------------+------------------------------------------+-------------------+------------+------------+-------------+----+------+-----------+----------+-----------------+-------------------+-------------------+----------------------+------------------+-------------+
|2025-12-26  |VJ0078a6f9e41d27c42ba12ee0c3d1f99b0052e788|10                 |0           |410000024371|0            |19  |49    |6          |0         |1                |0.0 

In [8]:
check_columns = [
    "published_line_name",
    "direction_id",
    "stop_id",
    "stop_sequence",
    "hour",
    "minute",
    "day_of_week",
    "is_weekend",
    "is_public_holiday",
    "journey_progress",
    "previous_stop_delay",
    "rolling_previous_delay",
    "has_previous_delay",
    "delay_seconds"
]

model_df.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in check_columns
]).show()

+-------------------+------------+-------+-------------+----+------+-----------+----------+-----------------+----------------+-------------------+----------------------+------------------+-------------+
|published_line_name|direction_id|stop_id|stop_sequence|hour|minute|day_of_week|is_weekend|is_public_holiday|journey_progress|previous_stop_delay|rolling_previous_delay|has_previous_delay|delay_seconds|
+-------------------+------------+-------+-------------+----+------+-----------+----------+-----------------+----------------+-------------------+----------------------+------------------+-------------+
|                  0|           0|      0|            0|   0|     0|          0|         0|                0|               0|                  0|                     0|                 0|            0|
+-------------------+------------+-------+-------------+----+------+-----------+----------+-----------------+----------------+-------------------+----------------------+------------------+

In [9]:
OUTPUT_PATH = r"D:\Big Data Programming Project\Final Assignment\data\processed\spark\scne_model_features"

model_df.write \
    .mode("overwrite") \
    .parquet(OUTPUT_PATH)

print("Saved feature dataset to:")
print(OUTPUT_PATH)

print("Saved rows:", spark.read.parquet(OUTPUT_PATH).count())

Saved feature dataset to:
D:\Big Data Programming Project\Final Assignment\data\processed\spark\scne_model_features
Saved rows: 308885
